# LFM Foundation: How Waves Create Gravity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Foundation_1D_Start_Here.ipynb)

## What This Notebook Demonstrates

Starting from just **two wave equations** (GOV-01 and GOV-02), we show that:

1. Wave packets propagate through a uniform substrate (the $\chi$ field)
2. Energy density creates **wells** in the $\chi$ field (via GOV-02)
3. These $\chi$-wells curve subsequent waves \u2014 **this IS gravity**

No Newton\u2019s law. No Einstein field equations. Just waves and substrate response.

---

## The LFM Governing Equations

**GOV-01** (Wave equation with position-dependent mass):

$$\frac{\partial^2 E}{\partial t^2} = c^2 \nabla^2 E - \chi^2 E$$

**GOV-02** (Substrate response to energy density):

$$\frac{\partial^2 \chi}{\partial t^2} = c^2 \nabla^2 \chi - \kappa (E^2 - E_0^2)$$

| Parameter | Value | Derivation |
|-----------|-------|------------|
| $\chi_0$ | 19 | $3^3 - 2^3$ = non-corner modes on 3D lattice |
| $\kappa$ | 1/63 | $1/(4^3 - 1)$ = inverse physical mode count |
| $c$ | 1 | Natural units |

**Key insight**: High energy density (large $E^2$) drives $\chi$ DOWN, creating a potential well. Waves curve toward low-$\chi$ regions. This *is* gravity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# LFM fundamental parameters (all derived, none fitted)
CHI_0 = 19.0          # vacuum stiffness: 3^3 - 2^3
KAPPA = 1.0 / 63.0    # coupling: 1/(4^3 - 1)
C = 1.0               # wave speed (natural units)

def laplacian_1d(f, dx):
    """1D discrete Laplacian with zero (absorbing) boundaries."""
    lap = np.zeros_like(f)
    lap[1:-1] = (f[2:] + f[:-2] - 2*f[1:-1]) / dx**2
    return lap

print("LFM Parameters:")
print(f"  chi_0 = {CHI_0}  (vacuum stiffness)")
print(f"  kappa = {KAPPA:.6f}  (coupling)")
print(f"  c     = {C}  (wave speed)")

## Run Simulation

We evolve a Gaussian wave packet on a 1D grid with coupled GOV-01 + GOV-02 dynamics.

In [ ]:
# Grid setup
N = 512
dx = 1.0
dt = 0.05
x = np.arange(N) * dx
n_steps = 2000

# Initialize: Gaussian wave packet at x ~ 128
E = 1.0 * np.exp(-(x - 128)**2 / (2 * 15**2)) * np.cos(0.5 * x)
E_prev = E.copy()
chi = np.ones(N) * CHI_0
chi_prev = chi.copy()

# Collect snapshots
snap_steps = [0, 300, 800, 2000]
E_snaps = {0: E.copy()}
chi_snaps = {0: chi.copy()}

for step in range(1, n_steps + 1):
    # GOV-01: wave propagation with chi-dependent mass
    E_next = 2*E - E_prev + dt**2 * (C**2 * laplacian_1d(E, dx) - chi**2 * E)
    # GOV-02: substrate response to energy density
    chi_next = 2*chi - chi_prev + dt**2 * (C**2 * laplacian_1d(chi, dx) - KAPPA * E**2)
    chi_next = np.maximum(chi_next, 0.1)  # prevent chi from going negative

    E_prev, E = E, E_next
    chi_prev, chi = chi, chi_next

    if step in snap_steps:
        E_snaps[step] = E.copy()
        chi_snaps[step] = chi.copy()

chi_dip = CHI_0 - chi.min()
print(f"Simulation complete: {n_steps} steps")
print(f"chi range: [{chi.min():.4f}, {chi.max():.4f}]  (vacuum = {CHI_0})")
print(f"Maximum chi-well depth: {chi_dip:.4f}")
if chi_dip > 0.001:
    print(f"\nENERGY DENSITY CREATED A chi-WELL -- THIS IS GRAVITY!")

In [ ]:
fig, axes = plt.subplots(len(snap_steps), 2, figsize=(14, 3 * len(snap_steps)))
fig.suptitle('LFM Foundation: Wave Energy Creates Gravitational Wells',
             fontsize=15, fontweight='bold', y=1.02)

for i, step in enumerate(snap_steps):
    t = step * dt
    axes[i, 0].plot(x, E_snaps[step], 'b-', linewidth=0.8)
    axes[i, 0].set_ylabel('E(x)')
    axes[i, 0].set_title(f'Wave Field E at t = {t:.0f}')
    axes[i, 0].set_ylim(-3, 3)
    axes[i, 0].grid(alpha=0.3)

    axes[i, 1].plot(x, chi_snaps[step], 'r-', linewidth=0.8)
    axes[i, 1].axhline(CHI_0, color='gray', ls='--', alpha=0.5,
                        label=f'chi_0 = {CHI_0}')
    axes[i, 1].set_ylabel('chi(x)')
    axes[i, 1].set_title(f'Substrate chi at t = {t:.0f}')
    axes[i, 1].grid(alpha=0.3)
    if i == 0:
        axes[i, 1].legend()

axes[-1, 0].set_xlabel('Position x')
axes[-1, 1].set_xlabel('Position x')
plt.tight_layout()
plt.show()

print("Left column: the E wave field propagating and dispersing")
print("Right column: the chi substrate developing wells where energy concentrates")

## Key Result

- **GOV-01** propagates wave energy across the lattice
- **GOV-02** responds to energy density by lowering $\chi$
- The $\chi$-well persists and deepens where energy concentrates
- Subsequent waves naturally curve toward low-$\chi$ regions

**This is gravity emerging from pure wave dynamics.** No external force law was injected.